# Pairwise Overmerge Signal — Pure SQL

Compute pairwise cosine similarity between all works per author, entirely in SQL.
No Python, no collect, no HDBSCAN — just SQL on the SQL warehouse / Photon.

Uses pre-staged work_ids from `temp_hdbscan_work_ids` (984 gold standard authors, capped 500 works).
Output: `pairwise_overmerge_signal` table with p10, p5, mean, min, stddev per author.

In [ ]:
import time

OUTPUT_TABLE = 'openalex.aer.pairwise_overmerge_signal'
STAGED_TABLE = 'openalex.aer.temp_hdbscan_work_ids'
EMBEDDINGS_TABLE = 'openalex.vector_search.work_embeddings_v2'
BATCH_SIZE = 50

# Get all author IDs from staged table
all_authors = sorted([r.author_id for r in spark.sql(f'SELECT DISTINCT author_id FROM {STAGED_TABLE}').collect()])
print(f'Total authors: {len(all_authors)}')

# Setup output table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {OUTPUT_TABLE} (
        author_id BIGINT, n_pairs BIGINT, mean_sim DOUBLE, p10_sim DOUBLE,
        p5_sim DOUBLE, min_sim DOUBLE, std_sim DOUBLE
    )
""")

# Skip already-done
done = set(r.author_id for r in spark.sql(f'SELECT author_id FROM {OUTPUT_TABLE}').collect())
todo = [a for a in all_authors if a not in done]
print(f'TODO: {len(todo)} ({len(done)} already done)')
print('=' * 80)

start = time.time()
processed = 0
errors = []

for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i+BATCH_SIZE]
    ids_str = ','.join(str(a) for a in batch)
    bt = time.time()

    try:
        spark.sql(f"""
            INSERT INTO {OUTPUT_TABLE}
            WITH author_works AS (
                SELECT t.author_id, t.work_id, e.embedding
                FROM {STAGED_TABLE} t
                JOIN {EMBEDDINGS_TABLE} e ON t.work_id = e.work_id
                WHERE t.author_id IN ({ids_str})
                  AND e.embedding IS NOT NULL
            ),
            pairs AS (
                SELECT
                    a.author_id,
                    aggregate(
                        transform(sequence(0, 1023), i -> a.embedding[i] * b.embedding[i]),
                        0.0D, (acc, v) -> acc + v
                    ) / (
                        sqrt(aggregate(transform(a.embedding, v -> v*v), 0.0D, (acc,v) -> acc+v)) *
                        sqrt(aggregate(transform(b.embedding, v -> v*v), 0.0D, (acc,v) -> acc+v))
                    ) AS cos_sim
                FROM author_works a
                JOIN author_works b ON a.author_id = b.author_id AND a.work_id < b.work_id
            )
            SELECT
                author_id,
                COUNT(*) AS n_pairs,
                AVG(cos_sim) AS mean_sim,
                PERCENTILE(cos_sim, 0.10) AS p10_sim,
                PERCENTILE(cos_sim, 0.05) AS p5_sim,
                MIN(cos_sim) AS min_sim,
                STDDEV(cos_sim) AS std_sim
            FROM pairs
            GROUP BY author_id
        """)
    except Exception as e:
        errors.append(f'Batch {i}: {e}')

    processed += len(batch)
    elapsed = time.time() - start
    rate = processed / elapsed if elapsed > 0 else 0
    eta = (len(todo) - processed) / rate if rate > 0 else 0
    bt_elapsed = time.time() - bt
    print(f'  [{processed}/{len(todo)}] batch {bt_elapsed:.1f}s | {elapsed:.0f}s elapsed, ETA {eta:.0f}s | {len(errors)} errors')

total_done = spark.sql(f'SELECT COUNT(*) AS n FROM {OUTPUT_TABLE}').collect()[0]['n']
print('=' * 80)
print(f'Done: {total_done} authors, {len(errors)} errors, {time.time()-start:.0f}s')

## Results

In [ ]:
import numpy as np

# Load gold standard labels
SPLIT_IDS = {5000058354,5000277470,5001106162,5001242576,5001751485,5002547846,5002981530,5002985815,5003051525,5003683328,5004699890,5004799011,5005483219,5005776483,5006626689,5006832353,5007114677,5007831897,5008757192,5008769348,5008979025,5009105221,5009137334,5009307852,5010422511,5010484703,5010975613,5011325322,5011900778,5012498525,5012689737,5013217186,5013478230,5013710631,5013938260,5014188192,5014594712,5015401837,5015773218,5016882470,5017142455,5018155146,5018218917,5019217807,5019862321,5020336126,5021173676,5021967634,5023003882,5024177610,5025452752,5027058224,5027417114,5027430243,5028103753,5029147304,5029904482,5030319607,5030455935,5031476162,5032690462,5033771456,5035071631,5036486793,5037136506,5037435179,5037784680,5037835036,5037924045,5038742031,5041864386,5043303728,5044626862,5044690856,5046337320,5046389819,5046438558,5048151405,5049568847,5049695364,5050939922,5051189473,5051468011,5052372225,5052571385,5052604833,5053863284,5054190140,5054499499,5054609123,5055122935,5055163904,5055757812,5056229359,5056502482,5056945568,5057408903,5057951893,5058036156,5058109631,5058421597,5059488532,5059871403,5059901313,5059938079,5060030012,5060512575,5061138371,5061782757,5061844230,5062125091,5062220638,5062516991,5063600613,5063896503,5064480798,5064679635,5065096990,5065410010,5065616148,5067290858,5067703574,5068211137,5068557265,5069506861,5070019932,5070168570,5070293380,5071183569,5071539405,5072305232,5073218194,5073480387,5074059264,5074251636,5074303692,5074432852,5074654930,5074948050,5075292110,5075755572,5076493708,5076836352,5078050424,5079166689,5079267101,5079413276,5079617160,5079632937,5079652777,5079979698,5080312909,5085619921,5087117526,5088152507,5091442005,5091859986,5100691568,5100775309,5101092520,5101114080,5101441706,5101536066,5101597741,5101672746,5101690769,5101833503,5101839347,5101989779,5102111335,5102715506,5102749436,5102755488,5103173829,5103347958,5103415285,5103459170,5103825320,5103825950,5103853738,5103907138,5103976114,5104029031,5106037625,5106467052,5107007469,5108376978,5108435508,5108453412,5108515912,5108570566,5108613657,5108623528,5108748587,5109066197,5109085753,5109148503,5109293548,5109296219,5109831141,5110027945,5110119960,5110150920,5110196385,5110316420,5110358878,5110393625,5110728098,5110749239,5110765172,5110897558,5111372795,5111461672,5111552276,5111574283,5111793979,5111812935,5112257425,5112300394,5112331881,5112390495,5112522951,5112655149,5112702198,5112723043,5113171091,5113370306,5113385911,5113434226,5113550326,5113625884,5113661926,5113857253,5113901982,5113952815,5113972030,5114021691,5114098473,5114339992,5117075932}

rows = [r.asDict() for r in spark.sql(f'SELECT * FROM {OUTPUT_TABLE}').collect()]
for r in rows:
    r['label'] = 'split' if r['author_id'] in SPLIT_IDS else 'confirmed'

output_lines = [f'Authors: {len(rows)}']
output_lines.append('')
output_lines.append('METRIC COMPARISON (median split vs confirmed):')
output_lines.append('=' * 80)

for metric in ['mean_sim', 'p10_sim', 'p5_sim', 'min_sim', 'std_sim']:
    sv = [r[metric] for r in rows if r['label'] == 'split' and r[metric] is not None]
    cv = [r[metric] for r in rows if r['label'] == 'confirmed' and r[metric] is not None]
    if sv and cv:
        sm, cm = np.median(sv), np.median(cv)
        gap = abs(sm - cm)
        ps = np.std(sv + cv)
        eff = gap / ps if ps > 0 else 0
        d = 'higher' if sm > cm else 'lower'
        output_lines.append(f'  {metric:15s}: split={sm:.4f} confirmed={cm:.4f} gap={gap:.4f} effect={eff:.2f} ({d})')

output_lines.append('')
for label in ['split', 'confirmed']:
    subset = [r for r in rows if r['label'] == label]
    output_lines.append(f'{label} (n={len(subset)}):')
    for metric in ['p10_sim', 'p5_sim', 'min_sim']:
        vals = [r[metric] for r in subset if r[metric] is not None]
        if vals:
            output_lines.append(f'  {metric}: median={np.median(vals):.4f} q1={np.percentile(vals,25):.4f} q3={np.percentile(vals,75):.4f}')

output = '\n'.join(output_lines)
print(output)
dbutils.notebook.exit(output)